In [6]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
REQUEST_TIMEOUT = 20

OUTPUT_DIR = Path(".")


def get_json(url: str, params: Optional[Dict[str, Any]] = None) -> Any:
    response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def parse_float(value: Any) -> Optional[float]:
    try:
        if value is None:
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def to_float_or_zero(value: Any) -> float:
    parsed = parse_float(value)
    return parsed if parsed is not None else 0.0


def get_event_by_slug(slug: str) -> Dict[str, Any]:
    slug = slug.strip()
    if not slug:
        raise ValueError("Slug cannot be empty.")
    return get_json(f"{GAMMA_API}/events/slug/{slug}")


def get_event_by_id(event_id: str) -> Dict[str, Any]:
    event_id = str(event_id).strip()
    if not event_id:
        raise ValueError("Event ID cannot be empty.")
    return get_json(f"{GAMMA_API}/events/{event_id}")


def find_markets_in_event(event: Dict[str, Any]) -> List[Dict[str, Any]]:
    for key in ("markets", "seriesMarkets", "children"):
        value = event.get(key)
        if isinstance(value, list):
            return value
    return []


def get_condition_id(market: Dict[str, Any]) -> Optional[str]:
    for key in ("conditionId", "condition_id", "condition"):
        value = market.get(key)
        if isinstance(value, str) and value:
            return value
    return None


def get_market_positions(
    condition_id: str,
    limit: int = 500,
    offset: int = 0,
    status: str = "ALL",
    sort_by: str = "TOTAL_PNL",
    sort_direction: str = "DESC",
) -> List[Dict[str, Any]]:
    return get_json(
        f"{DATA_API}/v1/market-positions",
        params={
            "market": condition_id,
            "status": status,
            "sortBy": sort_by,
            "sortDirection": sort_direction,
            "limit": limit,
            "offset": offset,
        },
    )


def normalize_position(position: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "proxyWallet": position.get("proxyWallet"),
        "name": position.get("name"),
        "asset": position.get("asset"),
        "size": position.get("size"),
        "avgPrice": position.get("avgPrice"),
        "currPrice": position.get("currPrice"),
        "currentValue": position.get("currentValue"),
        "cashPnl": position.get("cashPnl"),
        "totalBought": position.get("totalBought"),
        "realizedPnl": position.get("realizedPnl"),
        "totalPnl": position.get("totalPnl"),
        "outcome": position.get("outcome"),
        "outcomeIndex": position.get("outcomeIndex"),
        "verified": position.get("verified"),
        "profileImage": position.get("profileImage"),
        "conditionId": position.get("conditionId"),
    }


def rank_positions(
    positions: List[Dict[str, Any]],
    rank_by: str = "totalBought",
    top_n: int = 50,
) -> List[Dict[str, Any]]:
    ranked = sorted(
        positions,
        key=lambda x: to_float_or_zero(x.get(rank_by)),
        reverse=True,
    )
    return ranked[:top_n]


def split_yes_no(
    position_groups: List[Dict[str, Any]],
    rank_by: str = "totalBought",
    top_n: int = 50,
) -> Dict[str, List[Dict[str, Any]]]:
    yes_positions: List[Dict[str, Any]] = []
    no_positions: List[Dict[str, Any]] = []

    for group in position_groups:
        positions = group.get("positions", []) or []
        for pos in positions:
            normalized = normalize_position(pos)
            outcome = str(normalized.get("outcome", "")).strip().upper()

            if outcome == "YES":
                yes_positions.append(normalized)
            elif outcome == "NO":
                no_positions.append(normalized)

    return {
        "YES": rank_positions(yes_positions, rank_by=rank_by, top_n=top_n),
        "NO": rank_positions(no_positions, rank_by=rank_by, top_n=top_n),
    }


def extract_top_positions_for_event(
    event_id: str,
    rank_by: str = "totalBought",
    top_n: int = 50,
) -> Dict[str, Any]:
    event = get_event_by_id(event_id)
    event_title = event.get("title") or event.get("question") or f"Event {event_id}"
    event_slug = event.get("slug")
    markets = find_markets_in_event(event)

    if not markets:
        raise ValueError("No markets found inside the event response.")

    results: Dict[str, Any] = {
        "event_id": str(event_id),
        "event_slug": event_slug,
        "event_title": event_title,
        "rank_by": rank_by,
        "markets": [],
    }

    for idx, market in enumerate(markets, start=1):
        condition_id = get_condition_id(market)
        if not condition_id:
            continue

        question = market.get("question") or market.get("title") or f"Market {idx}"
        market_id = market.get("id")
        market_slug = market.get("slug")

        groups = get_market_positions(
            condition_id=condition_id,
            limit=500,
            offset=0,
            status="ALL",
            sort_by="TOTAL_PNL",
            sort_direction="DESC",
        )

        split = split_yes_no(groups, rank_by=rank_by, top_n=top_n)

        results["markets"].append(
            {
                "market_id": market_id,
                "market_slug": market_slug,
                "market_question": question,
                "condition_id": condition_id,
                "top_yes": split["YES"],
                "top_no": split["NO"],
            }
        )

    return results


def get_user_leaderboard_stats(user: str) -> Dict[str, float]:
    rows = get_json(
        f"{DATA_API}/v1/leaderboard",
        params={
            "user": user,
            "timePeriod": "ALL",
            "orderBy": "VOL",
            "limit": 1,
            "offset": 0,
        },
    )

    if isinstance(rows, list) and rows:
        row = rows[0]
        return {
            "account_total_traded_volume": to_float_or_zero(row.get("vol")),
            "account_total_gain_loss": to_float_or_zero(row.get("pnl")),
        }

    return {
        "account_total_traded_volume": 0.0,
        "account_total_gain_loss": 0.0,
    }


def build_positions_dataframe(data: Dict[str, Any]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for market in data.get("markets", []):
        market_question = market.get("market_question", "")
        market_id = market.get("market_id", "")
        market_slug = market.get("market_slug", "")
        condition_id = market.get("condition_id", "")

        for side_key, side_name in (("top_yes", "YES"), ("top_no", "NO")):
            holders = market.get(side_key, []) or []
            for rank, holder in enumerate(holders, start=1):
                rows.append(
                    {
                        "event_id": data.get("event_id", ""),
                        "event_slug": data.get("event_slug", ""),
                        "event_title": data.get("event_title", ""),
                        "market_id": market_id,
                        "market_slug": market_slug,
                        "market_question": market_question,
                        "condition_id": condition_id,
                        "side": side_name,
                        "rank": rank,
                        "name": holder.get("name") or "",
                        "proxyWallet": holder.get("proxyWallet") or "",
                        "verified": holder.get("verified"),
                        "asset": holder.get("asset") or "",
                        "size": to_float_or_zero(holder.get("size")),
                        "totalBought": to_float_or_zero(holder.get("totalBought")),
                        "avgPrice": to_float_or_zero(holder.get("avgPrice")),
                        "currPrice": to_float_or_zero(holder.get("currPrice")),
                        "currentValue": to_float_or_zero(holder.get("currentValue")),
                        "cashPnl": to_float_or_zero(holder.get("cashPnl")),
                        "realizedPnl": to_float_or_zero(holder.get("realizedPnl")),
                        "totalPnl": to_float_or_zero(holder.get("totalPnl")),
                        "outcomeIndex": holder.get("outcomeIndex"),
                    }
                )

    return pd.DataFrame(rows)


def enrich_dataframe_with_account_totals(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "proxyWallet" not in df.columns:
        return df

    unique_wallets = [w for w in df["proxyWallet"].dropna().astype(str).unique() if w]
    totals_by_wallet: Dict[str, Dict[str, float]] = {}

    for i, wallet in enumerate(unique_wallets, start=1):
        print(f"Enriching account {i}/{len(unique_wallets)}: {wallet}")
        try:
            totals_by_wallet[wallet] = get_user_leaderboard_stats(wallet)
        except Exception as e:
            print(f"  Failed for {wallet}: {e}")
            totals_by_wallet[wallet] = {
                "account_total_traded_volume": 0.0,
                "account_total_gain_loss": 0.0,
            }

    enriched = df.copy()
    enriched["account_total_traded_volume"] = enriched["proxyWallet"].map(
        lambda w: totals_by_wallet.get(str(w), {}).get("account_total_traded_volume", 0.0)
    )
    enriched["account_total_gain_loss"] = enriched["proxyWallet"].map(
        lambda w: totals_by_wallet.get(str(w), {}).get("account_total_gain_loss", 0.0)
    )

    return enriched


def build_winners_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Winner rows only:
    realizedPnl > 0 is used as the filter.
    Account enrichment is applied only to these rows.
    """
    if df.empty:
        return df.copy()

    winners = df[df["realizedPnl"] > 0].copy()

    if winners.empty:
        return winners

    winners = winners.sort_values(
        ["realizedPnl", "totalPnl", "totalBought"],
        ascending=False,
    ).reset_index(drop=True)

    winners = enrich_dataframe_with_account_totals(winners)
    return winners


def save_highlighted_table_html(
    df: pd.DataFrame,
    output_file: Path,
    title: str,
    subtitle: str,
    highlight_columns: Optional[List[str]] = None,
) -> None:
    if df.empty:
        output_file.write_text(
            f"<html><body><h2>{title}</h2><p>No data available</p></body></html>",
            encoding="utf-8",
        )
        return

    if highlight_columns is None:
        highlight_columns = [
            "totalBought",
            "currentValue",
            "realizedPnl",
            "totalPnl",
            "account_total_traded_volume",
            "account_total_gain_loss",
        ]

    format_map = {
        "size": "{:,.2f}",
        "totalBought": "{:,.2f}",
        "avgPrice": "{:,.4f}",
        "currPrice": "{:,.4f}",
        "currentValue": "{:,.2f}",
        "cashPnl": "{:,.2f}",
        "realizedPnl": "{:,.2f}",
        "totalPnl": "{:,.2f}",
        "account_total_traded_volume": "{:,.2f}",
        "account_total_gain_loss": "{:,.2f}",
    }

    styler = df.style.format(format_map)

    for col in highlight_columns:
        if col in df.columns:
            styler = styler.background_gradient(cmap="Reds", subset=[col])

    styler = styler.set_properties(
        **{
            "text-align": "left",
            "border": "1px solid #d0d0d0",
            "font-size": "13px",
            "padding": "6px",
        }
    ).set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                    ("font-family", "Arial, sans-serif"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f5f5f5"),
                    ("border", "1px solid #d0d0d0"),
                    ("padding", "6px"),
                    ("position", "sticky"),
                    ("top", "0"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #d0d0d0"),
                    ("padding", "6px"),
                ],
            },
        ]
    )

    html = f"""
    <html>
    <head>
        <meta charset="utf-8">
        <title>{title}</title>
    </head>
    <body>
        <h2>{title}</h2>
        <p>{subtitle}</p>
        {styler.to_html()}
    </body>
    </html>
    """

    output_file.write_text(html, encoding="utf-8")


def print_summary(data: Dict[str, Any], df: pd.DataFrame, winners_df: pd.DataFrame) -> None:
    print(f"Event title: {data.get('event_title')}")
    print(f"Event slug:  {data.get('event_slug')}")
    print(f"Event ID:    {data.get('event_id')}")
    print(f"Ranked by:   {data.get('rank_by')}")
    print(f"All rows:    {len(df)}")
    print(f"Winner rows: {len(winners_df)}")
    print("=" * 90)

    for market in data.get("markets", []):
        print(f"\nMarket: {market.get('market_question')}")
        print(f"Market ID: {market.get('market_id')}")
        print(f"Condition ID: {market.get('condition_id')}")
        print(f"Top YES rows: {len(market.get('top_yes', []))}")
        print(f"Top NO rows:  {len(market.get('top_no', []))}")


def main() -> None:
    slug = input("Enter Polymarket event slug: ").strip()

    event = get_event_by_slug(slug)
    event_id = str(event["id"])

    print("\nResolved slug to event:")
    print(f"Title: {event.get('title')}")
    print(f"Slug:  {event.get('slug')}")
    print(f"ID:    {event_id}")

    rank_by = "totalBought"

    data = extract_top_positions_for_event(
        event_id=event_id,
        rank_by=rank_by,
        top_n=50,
    )

    df = build_positions_dataframe(data)

    # Main table: no account enrichment
    top_json_file = OUTPUT_DIR / "top_positions_by_event.json"
    top_csv_file = OUTPUT_DIR / "top_positions_by_event.csv"
    top_html_file = OUTPUT_DIR / "top_positions_table.html"

    with top_json_file.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    df.to_csv(top_csv_file, index=False, encoding="utf-8")
    save_highlighted_table_html(
        df=df,
        output_file=top_html_file,
        title="Top Historical Positions by Event",
        subtitle="Includes open and closed positions. Darker red indicates larger values.",
        highlight_columns=["totalBought", "currentValue", "realizedPnl", "totalPnl"],
    )

    # Winners analysis: enrich only winner rows
    winners_df = build_winners_analysis(df)

    winners_csv_file = OUTPUT_DIR / "winners_analysis.csv"
    winners_html_file = OUTPUT_DIR / "winners_analysis.html"

    winners_df.to_csv(winners_csv_file, index=False, encoding="utf-8")
    save_highlighted_table_html(
        df=winners_df,
        output_file=winners_html_file,
        title="Winners Analysis",
        subtitle="Only rows with realizedPnl > 0. Includes wallet-level traded volume and gain/loss from leaderboard data.",
        highlight_columns=[
            "realizedPnl",
            "totalPnl",
            "totalBought",
            "account_total_traded_volume",
            "account_total_gain_loss",
        ],
    )

    print()
    print_summary(data, df, winners_df)
    print("\nSaved files:")
    print(f"- {top_json_file}")
    print(f"- {top_csv_file}")
    print(f"- {top_html_file}")
    print(f"- {winners_csv_file}")
    print(f"- {winners_html_file}")


if __name__ == "__main__":
    main()


Resolved slug to event:
Title: US strikes Iran by...?
Slug:  us-strikes-iran-by
ID:    114242
Enriching account 1/1631: 0x1caa6a7ad0c6916aef7b67946de2e57ad24846a0
Enriching account 2/1631: 0xbacd00c9080a82ded56f504ee8810af732b0ab35
Enriching account 3/1631: 0x5ed3561e39d6c9a9622c6f84a782537df7f9b19e
Enriching account 4/1631: 0x33acd201d6b02982b3a95b975b331bc225649334
Enriching account 5/1631: 0xdbf16bcffa41bd95faec2640daa2c95ab2bd86b5
Enriching account 6/1631: 0x0a854897a06d4999e5b2dde5693609f1428ffe9d
Enriching account 7/1631: 0x09d3273fa76282ce09f4f35a87d6f087c05f4e84
Enriching account 8/1631: 0x607fec58640fcea5cb837adb09d658747720592c
Enriching account 9/1631: 0x37d10ffb61998561c5f9fb941c42c952d8fb4e28
Enriching account 10/1631: 0x514550b8f94f24e3bd5fca1c8af19e985fb71f46
Enriching account 11/1631: 0x8c80d213c0cbad777d06ee3f58f6ca4bc03102c3
Enriching account 12/1631: 0x11e50ec01d48adc0be2292cb8e2a5fee0369ee4d
Enriching account 13/1631: 0xde7be6d489bce070a959e0cb813128ae659b5f4b
Enri